# Using user defined functions in Spark

- You've seen some of the power behind Spark's built-in string functions when it comes to manipulating DataFrames. However, once you reach a certain point, it becomes difficult to process the data in a without creating a rat's nest of function calls. Here's one place where you can use User Defined Functions to manipulate our DataFrames.

- For this exercise, we'll use our `voter_df` DataFrame, but you're going to replace the `first_name` column with the first and middle names.

- The `pyspark.sql.functions` library is available under the alias `F`. The classes from `pyspark.sql.types` are already imported.

## Instructions

- Edit the `getFirstAndMiddle()` function to return a space separated string of names, except the last entry in the names list.
- Define the function as a user-defined function. It should return a string type.
- Create a new column on `voter_df` called `first_and_middle_name` using your UDF.
- Show the Data Frame.

In [1]:
# Explicitly Point Spark to the Correct Python Executable
# If you already have numpy installed in a specific environment (like conda or a virtualenv), but PySpark is defaulting to /usr/bin/python3 or system Python, 
# explicitly set the Python environment variables before initializing SparkContext:

import os
import sys

# Set these before initializing SparkContext (sc)
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
from pyspark.sql import SparkSession

In [3]:
# If you are developing an application you should use local mode. 
# Running in yarn mode is bit time consuming than running in local mode
# If you still insist to run in yarn mode then make sure that
# 1. You assigned more than one processor in the VM settings
# 2. Run only one kernel at a time

In [4]:
# spark = SparkSession. \
#     builder. \
#     enableHiveSupport(). \
#     appName('Demo'). \
#     master('yarn'). \
#     getOrCreate()

In [5]:
spark = SparkSession. \
    builder. \
    enableHiveSupport(). \
    appName('Demo'). \
    master('local[*]'). \
    getOrCreate()

In [6]:
sc = spark.sparkContext

In [21]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

# Load the CSV file
voter_df = spark.read.format('csv').options(Header=True).load('file:///home/talentum/UBUNTU_HOME/test-jupyter/P3/M2/SM3/3_UserDefinedFunctions/Dataset/DallasCouncilVoters.csv.gz')

# Add a new column called splits separated on whitespace
voter_df = voter_df.withColumn('splits', F.split(voter_df.VOTER_NAME, '\s+'))

# Create a new column called first_name based on the first item in splits
voter_df = voter_df.withColumn('first_name', voter_df.splits.getItem(0))

# Get the last entry of the splits list and create a column called last_name
voter_df = voter_df.withColumn('last_name', voter_df.splits.getItem(F.size('splits') - 1))

def getFirstAndMiddle(names):
  # Return a space separated string of names
    if not names:
        return ''
    return ' '.join(names[:-1])

# Define the method as a UDF
udfFirstAndMiddle = F.udf(getFirstAndMiddle, StringType())

# Create a new column using your UDF
voter_df = voter_df.withColumn('first_and_middle_name', udfFirstAndMiddle(voter_df.splits))

# Show the DataFrame
voter_df.show()


+----------+-------------+-------------------+--------------------+----------+---------+---------------------+
|      DATE|        TITLE|         VOTER_NAME|              splits|first_name|last_name|first_and_middle_name|
+----------+-------------+-------------------+--------------------+----------+---------+---------------------+
|02/08/2017|Councilmember|  Jennifer S. Gates|[Jennifer, S., Ga...|  Jennifer|    Gates|          Jennifer S.|
|02/08/2017|Councilmember| Philip T. Kingston|[Philip, T., King...|    Philip| Kingston|            Philip T.|
|02/08/2017|        Mayor|Michael S. Rawlings|[Michael, S., Raw...|   Michael| Rawlings|           Michael S.|
|02/08/2017|Councilmember|       Adam Medrano|     [Adam, Medrano]|      Adam|  Medrano|                 Adam|
|02/08/2017|Councilmember|       Casey Thomas|     [Casey, Thomas]|     Casey|   Thomas|                Casey|
|02/08/2017|Councilmember|Carolyn King Arnold|[Carolyn, King, A...|   Carolyn|   Arnold|         Carolyn King|
|

In [26]:
import pyspark.sql.functions as F

voter_df_null = voter_df.filter(F.isnull('VOTER_NAME'))
voter_df_null.show()

null_count = voter_df_null.count()
print(null_count)

total_count = voter_df.count()

null_voter_percentage = (null_count/total_count) * 100

print(null_voter_percentage)

+--------------------+------+----------+------+----------+---------+---------------------+
|                DATE| TITLE|VOTER_NAME|splits|first_name|last_name|first_and_middle_name|
+--------------------+------+----------+------+----------+---------+---------------------+
|          08/15/2018|Vacant|      null|  null|      null|     null|                     |
|          08/15/2018|Vacant|      null|  null|      null|     null|                     |
|          09/18/2018|Vacant|      null|  null|      null|     null|                     |
|          09/12/2018|Vacant|      null|  null|      null|     null|                     |
|[APPOINTMENT OF J...|  null|      null|  null|      null|     null|                     |
|[APPOINTMENT OF J...|  null|      null|  null|      null|     null|                     |
|[APPOINTMENT OF J...|  null|      null|  null|      null|     null|                     |
|[APPOINTMENT OF J...|  null|      null|  null|      null|     null|                     |

In [27]:
voter_df = spark.read.format('csv').options(Header=True).load('file:///home/talentum/UBUNTU_HOME/test-jupyter/P3/M2/SM3/3_UserDefinedFunctions/Dataset/DallasCouncilVoters.csv.gz')

# Replace null values in VOTER_NAME with 'John Doe'
voter_df = voter_df.fillna({'VOTER_NAME': 'John Doe'})

# Verify the changes by showing rows where the name was replaced
voter_df.filter(voter_df.VOTER_NAME == 'John Doe').show(10, truncate=False)

+--------------------------------+------+----------+
|DATE                            |TITLE |VOTER_NAME|
+--------------------------------+------+----------+
|08/15/2018                      |Vacant|John Doe  |
|08/15/2018                      |Vacant|John Doe  |
|09/18/2018                      |Vacant|John Doe  |
|09/12/2018                      |Vacant|John Doe  |
|[APPOINTMENT OF JOSELITO GIRON]"|null  |John Doe  |
|[APPOINTMENT OF JOSELITO GIRON]"|null  |John Doe  |
|[APPOINTMENT OF JOSELITO GIRON]"|null  |John Doe  |
|[APPOINTMENT OF JOSELITO GIRON]"|null  |John Doe  |
|[APPOINTMENT OF JOSELITO GIRON]"|null  |John Doe  |
|[APPOINTMENT OF JOSELITO GIRON]"|null  |John Doe  |
+--------------------------------+------+----------+
only showing top 10 rows



In [42]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

# Load the CSV file
voter_df = spark.read.format('csv').options(Header=True).load('file:///home/talentum/UBUNTU_HOME/test-jupyter/P3/M2/SM3/3_UserDefinedFunctions/Dataset/DallasCouncilVoters.csv.gz')

# Add a new column called splits separated on whitespace
voter_df = voter_df.withColumn('splits', F.split(voter_df.VOTER_NAME, '\s+'))


def getFirstAndMiddle(names):
  # Return a space separated string of names
    if not names:
        return ''
    return ' '.join(names[:-1])


def getFirst(names):
    
    if not names:
        return ''
    return names[0]

def getMiddle(names):
    
    if not names or len(names) != 3:
        return ''
    return names[1]

def getLast(names):
    
    if not names:
        return ''
    return names[-1]

# Define the method as a UDF
udfFirstAndMiddle = F.udf(getFirstAndMiddle, StringType())


udfFirst = F.udf(getFirst, StringType())

udfMiddle = F.udf(getMiddle, StringType())

udfLast = F.udf(getLast, StringType())

# Create a new column using your UDF
voter_df = voter_df.withColumn('first_and_middle_name', udfFirstAndMiddle(voter_df.splits))

voter_df = voter_df.withColumn('first_name', udfFirst(voter_df.splits))\
            .withColumn('middle_name', udfMiddle(voter_df.splits))\
            .withColumn('last_name', udfLast(voter_df.splits)).drop('splits').show()


+----------+-------------+-------------------+---------------------+----------+-----------+---------+
|      DATE|        TITLE|         VOTER_NAME|first_and_middle_name|first_name|middle_name|last_name|
+----------+-------------+-------------------+---------------------+----------+-----------+---------+
|02/08/2017|Councilmember|  Jennifer S. Gates|          Jennifer S.|  Jennifer|         S.|    Gates|
|02/08/2017|Councilmember| Philip T. Kingston|            Philip T.|    Philip|         T.| Kingston|
|02/08/2017|        Mayor|Michael S. Rawlings|           Michael S.|   Michael|         S.| Rawlings|
|02/08/2017|Councilmember|       Adam Medrano|                 Adam|      Adam|           |  Medrano|
|02/08/2017|Councilmember|       Casey Thomas|                Casey|     Casey|           |   Thomas|
|02/08/2017|Councilmember|Carolyn King Arnold|         Carolyn King|   Carolyn|       King|   Arnold|
|02/08/2017|Councilmember|       Scott Griggs|                Scott|     Scott|   

# Caching Related Code

In [43]:
from pyspark.sql.functions import monotonically_increasing_id

# 1. Collect and print the mapped partitions list
print(sc.parallelize(range(2), 2)
      .mapPartitions(lambda x: [(1,), (2,), (3,)]).collect())

# 2. Create the DataFrame
df = sc.parallelize(range(2), 2) \
       .mapPartitions(lambda x: [(1,), (2,), (3,)]).toDF(['col1'])

# 3. Add the monotonically increasing ID
df = df.select('col1', monotonically_increasing_id().alias('id'))

# 4. Display the results
df.show()

[(1,), (2,), (3,), (1,), (2,), (3,)]
+----+----------+
|col1|        id|
+----+----------+
|   1|         0|
|   2|         1|
|   3|         2|
|   1|8589934592|
|   2|8589934593|
|   3|8589934594|
+----+----------+

